In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

True
NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
import os
import pandas as pd
from pathlib import Path

SHENZHEN_DIR = Path("/Dissertation/data/Shenzhen/images")
MONTGOMERY_DIR = Path("/Dissertation/data/Montgomery/images")
OUTPUT_CSV = "dataset.csv"
def get_label_from_filename(filename):
    """
    Shenzhen and Montgomery filenames usually end with _0 or _1 before extension.
    _0 = non-TB
    _1 = TB
    """
    stem = Path(filename).stem

    if stem.endswith("_0"):
        return 0
    elif stem.endswith("_1"):
        return 1
    else:
        raise ValueError(f"Cannot infer label from filename: {filename}")


def collect_images(folder, dataset_name):
    records = []

    for img_path in folder.glob("*"):
        if img_path.suffix.lower() in [".png", ".jpg", ".jpeg"]:
            label = get_label_from_filename(img_path.name)

            records.append({
                "image_path": str(img_path),
                "label": label,
                "dataset": dataset_name,
                "image_id": img_path.stem
            })

    return records


records = []
records.extend(collect_images(SHENZHEN_DIR, "Shenzhen"))
records.extend(collect_images(MONTGOMERY_DIR, "Montgomery"))

df = pd.DataFrame(records)

print(df.head())
print("\nDataset size:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts())
print("\nDataset distribution:")
print(df["dataset"].value_counts())

df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved dataset CSV to {OUTPUT_CSV}")

                                          image_path  label   dataset  \
0  OneDrive\Documents\Dissertation...      0  Shenzhen   
1  OneDrive\Documents\Dissertation...      0  Shenzhen   
2  OneDrive\Documents\Dissertation...      0  Shenzhen   
3  OneDrive\Documents\Dissertation...      0  Shenzhen   
4  OneDrive\Documents\Dissertation...      0  Shenzhen   

        image_id  
0  CHNCXR_0001_0  
1  CHNCXR_0002_0  
2  CHNCXR_0003_0  
3  CHNCXR_0004_0  
4  CHNCXR_0005_0  

Dataset size: 800

Class distribution:
label
0    406
1    394
Name: count, dtype: int64

Dataset distribution:
dataset
Shenzhen      662
Montgomery    138
Name: count, dtype: int64

Saved dataset CSV to dataset.csv


In [5]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
CSV_PATH = "dataset.csv"
OUTPUT_DIR = "outputs"
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
METRICS_DIR = os.path.join(OUTPUT_DIR, "metrics")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 25
LR = 1e-4
NUM_CLASSES = 2
PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Makes training more reproducible, though sometimes slower.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

# Dataset Class
class TBXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = row["image_path"]
        label = int(row["label"])

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label, row["image_id"], img_path

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load CSV and Split
df = pd.read_csv(CSV_PATH)
print("Total images:", len(df))
print(df["label"].value_counts())

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)
print("\nSplit sizes:")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

train_df.to_csv(os.path.join(OUTPUT_DIR, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_split.csv"), index=False)

# Weighted Sampling for Imbalance
class_counts = train_df["label"].value_counts().sort_index().values
class_weights = 1.0 / class_counts

sample_weights = train_df["label"].map({
    0: class_weights[0],
    1: class_weights[1]
}).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

# DataLoaders
train_dataset = TBXrayDataset(train_df, transform=train_transform)
val_dataset = TBXrayDataset(val_df, transform=val_test_transform)
test_dataset = TBXrayDataset(test_df, transform=val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# DenseNet121 Model
model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, NUM_CLASSES)
model = model.to(DEVICE)

# Loss and Optimiser
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# Evaluation Function
def evaluate(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []
    all_image_ids = []
    all_paths = []

    with torch.no_grad():
        for images, labels, image_ids, paths in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            running_loss += loss.item() * images.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_image_ids.extend(image_ids)
            all_paths.extend(paths)

    epoch_loss = running_loss / len(loader.dataset)

    return {
        "loss": epoch_loss,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs),
        "image_ids": all_image_ids,
        "paths": all_paths
    }

# Training Loop
best_val_loss = float("inf")
epochs_no_improve = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_auc": []
}

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels, _, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    val_results = evaluate(model, val_loader, criterion)
    val_loss = val_results["loss"]

    val_acc = accuracy_score(val_results["labels"], val_results["preds"])

    try:
        val_auc = roc_auc_score(val_results["labels"], val_results["probs"])
    except ValueError:
        val_auc = 0.0

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)
    history["val_auc"].append(val_auc)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0

        torch.save(model.state_dict(), os.path.join(MODEL_DIR, "best_densenet121_tb.pth"))
        print("Saved best model.")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print("Early stopping triggered.")
        break

# Plot Training Curves
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DenseNet121 Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "densenet121_loss_curve.png"))
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(history["val_accuracy"], label="Validation Accuracy")
plt.plot(history["val_auc"], label="Validation ROC-AUC")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("DenseNet121 Validation Accuracy and ROC-AUC")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "densenet121_val_metrics.png"))
plt.close()

# Test Evaluation
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "best_densenet121_tb.pth"), map_location=DEVICE))
test_results = evaluate(model, test_loader, criterion)

y_true = test_results["labels"]
y_pred = test_results["preds"]
y_prob = test_results["probs"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
sensitivity = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

try:
    roc_auc = roc_auc_score(y_true, y_prob)
except ValueError:
    roc_auc = 0.0

metrics = {
    "accuracy": accuracy,
    "precision": precision,
    "sensitivity_recall": sensitivity,
    "specificity": specificity,
    "f1_score": f1,
    "roc_auc": roc_auc,
    "true_negative": int(tn),
    "false_positive": int(fp),
    "false_negative": int(fn),
    "true_positive": int(tp)
}

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(os.path.join(METRICS_DIR, "densenet121_test_metrics.csv"), index=False)

print("\nTest Metrics:")
print(metrics_df)
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Non-TB", "TB"], zero_division=0))

# Save Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Pred Non-TB", "Pred TB"],
    yticklabels=["True Non-TB", "True TB"]
)
plt.title("DenseNet121 Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "densenet121_confusion_matrix.png"))
plt.close()

# Save CNN Prediction Probabilities for Phase 3
pred_df = pd.DataFrame({
    "image_id": test_results["image_ids"],
    "image_path": test_results["paths"],
    "true_label": y_true,
    "cnn_prediction": y_pred,
    "cnn_probability_tb": y_prob
})
pred_df.to_csv(os.path.join(PRED_DIR, "densenet121_test_predictions.csv"), index=False)
print("\nSaved test predictions for surrogate modelling.")

Using device: cuda
Total images: 800
label
0    406
1    394
Name: count, dtype: int64

Split sizes:
Train: 560
Val: 120
Test: 120


Epoch 1/25: 100%|██████████████████████████████████████████████████████████████████████| 35/35 [01:50<00:00,  3.16s/it]


Epoch 1/25 | Train Loss: 0.4658 | Val Loss: 0.3522 | Val Acc: 0.8917 | Val AUC: 0.9280
Saved best model.


Epoch 2/25: 100%|██████████████████████████████████████████████████████████████████████| 35/35 [01:27<00:00,  2.50s/it]


Epoch 2/25 | Train Loss: 0.3213 | Val Loss: 0.3628 | Val Acc: 0.8500 | Val AUC: 0.9325


...

Epoch 13/25 | Train Loss: 0.0560 | Val Loss: 0.2957 | Val Acc: 0.9083 | Val AUC: 0.9661


Epoch 14/25: 100%|█████████████████████████████████████████████████████████████████████| 35/35 [01:06<00:00,  1.90s/it]


Epoch 14/25 | Train Loss: 0.0476 | Val Loss: 0.2538 | Val Acc: 0.9167 | Val AUC: 0.9675


Epoch 15/25: 100%|█████████████████████████████████████████████████████████████████████| 35/35 [01:08<00:00,  1.96s/it]


Epoch 15/25 | Train Loss: 0.0339 | Val Loss: 0.2554 | Val Acc: 0.9000 | Val AUC: 0.9678
Early stopping triggered.

Test Metrics:
   accuracy  precision  sensitivity_recall  specificity  f1_score   roc_auc  \
0      0.85   0.836066            0.864407     0.836066      0.85  0.902195   

   true_negative  false_positive  false_negative  true_positive  
0             51              10               8             51  

Classification Report:
              precision    recall  f1-score   support

      Non-TB       0.86      0.84      0.85        61
          TB       0.84      0.86      0.85        59

    accuracy                           0.85       120
   macro avg       0.85      0.85      0.85       120
weighted avg       0.85      0.85      0.85       120


Saved test predictions for surrogate modelling.
